# Sampling Methods: Inverse Transform, Rejection & Importance

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/sampling-methods)

We sample from scratch with each method, measure the rejection acceptance rate, use importance sampling for a rare-event probability, and show antithetic / stratified variance reduction.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — Inverse transform sampling

For the exponential, $F^{-1}(u) = -\frac{1}{\lambda}\ln(1-u)$. One uniform draw -> one exact sample.

In [ ]:
lam = 1.5
u = rng.random(20000)
x = -np.log(1-u)/lam   # inverse CDF

plt.figure(figsize=(8,4))
plt.hist(x, bins=60, density=True, alpha=0.7, color='#6366f1', label='inverse-transform samples')
grid = np.linspace(0, 6, 200)
plt.plot(grid, lam*np.exp(-lam*grid), color='#f59e0b', lw=2, label='true Exp(1.5) density')
plt.legend(); plt.title('Inverse transform sampling'); plt.tight_layout(); plt.show()
print(f"sample mean {x.mean():.3f}  (theory 1/lambda = {1/lam:.3f})")

## 2 — Rejection sampling

Target: a bimodal density we can evaluate but not easily invert. Proposal: a wide Gaussian envelope.

In [ ]:
# Unnormalized bimodal target
p_tilde = lambda x: np.exp(-0.5*((x+2)/0.5)**2) + np.exp(-0.5*((x-2)/0.8)**2)
q = stats.norm(0, 2.5)              # proposal
M = (p_tilde(np.linspace(-8,8,2000)) / q.pdf(np.linspace(-8,8,2000))).max() * 1.05

def rejection_sample(n):
    accepted = []
    tries = 0
    while len(accepted) < n:
        cand = q.rvs(random_state=rng)
        tries += 1
        if rng.random() <= p_tilde(cand)/(M*q.pdf(cand)):
            accepted.append(cand)
    return np.array(accepted), n/tries

samples, acc_rate = rejection_sample(10000)
print(f"Acceptance rate = {acc_rate:.3f}  (theory 1/M with normalization; loose M wastes draws)")

grid = np.linspace(-6, 6, 400)
Z = np.trapz(p_tilde(grid), grid)   # normalizer for plotting
plt.figure(figsize=(8,4))
plt.hist(samples, bins=70, density=True, alpha=0.7, color='#6366f1', label='accepted samples')
plt.plot(grid, p_tilde(grid)/Z, color='#f59e0b', lw=2, label='target density')
plt.plot(grid, M*q.pdf(grid)/Z, color='#f87171', ls='--', lw=1, label='M*q envelope')
plt.legend(); plt.title('Rejection sampling'); plt.tight_layout(); plt.show()

## 3 — Importance sampling for a rare event

Estimate $P(X>4.5)$ for $X\sim N(0,1)$. Naive Monte Carlo almost never lands in the tail; a shifted proposal does, then reweights.

In [ ]:
truth = stats.norm.sf(4.5)   # true tail probability
N = 100000

# Naive Monte Carlo
xs = rng.normal(0, 1, N)
naive = (xs > 4.5).mean()

# Importance sampling: proposal N(4.5, 1)
p = stats.norm(0, 1); qd = stats.norm(4.5, 1)
xq = qd.rvs(N, random_state=rng)
w = p.pdf(xq)/qd.pdf(xq)
is_est = (( xq > 4.5).astype(float) * w).mean()

print(f"True P(X>4.5)        = {truth:.3e}")
print(f"Naive MC estimate    = {naive:.3e}   (often 0 — almost no samples in tail)")
print(f"Importance sampling  = {is_est:.3e}")
print(f"IS relative error    = {abs(is_est-truth)/truth:.2%}")

## 4 — Variance reduction: antithetic variates

Pair each u with 1-u; for a monotone integrand the pair is negatively correlated, halving variance.

In [ ]:
# Estimate E[e^U] for U~Uniform(0,1) (true value e-1)
true_val = np.e - 1
n_trials = 2000; n = 100
plain_var, anti_var = [], []
for _ in range(n_trials):
    u = rng.random(n)
    plain_var.append(np.exp(u).mean())
    u_half = rng.random(n//2)
    anti_var.append(np.concatenate([np.exp(u_half), np.exp(1-u_half)]).mean())
print(f"True E[e^U] = {true_val:.4f}")
print(f"Plain MC    : mean {np.mean(plain_var):.4f}, var {np.var(plain_var):.2e}")
print(f"Antithetic  : mean {np.mean(anti_var):.4f}, var {np.var(anti_var):.2e}")
print(f"Variance reduction factor: {np.var(plain_var)/np.var(anti_var):.1f}x")

## ✏️ Your turn

**Task A — Self-normalized importance sampling:** When the target is known only up to a constant, estimate $\mathbb{E}_p[f]$ as $\sum_i w_i f(x_i) / \sum_i w_i$. Use it to estimate the mean of the unnormalized bimodal target from section 2 with a Gaussian proposal, and inspect the weight distribution (are a few weights dominating?).

**Task B — Effective sample size of importance weights:** Implement $\text{ESS} = (\sum_i w_i)^2 / \sum_i w_i^2$. Show that as the proposal moves away from the target, ESS collapses toward 1 — the signal that importance sampling is failing.

In [ ]:
def snis_mean(p_tilde, proposal, n=20000):
    """Self-normalized importance-sampling estimate of E_p[x]."""
    x = proposal.rvs(n, random_state=rng)
    w = p_tilde(x) / proposal.pdf(x)
    # TODO(you): return the self-normalized weighted mean of x
    return ...

est = snis_mean(p_tilde, stats.norm(0, 3))
if est is not None:
    print(f"SNIS estimate of target mean = {est:.4f}")

<details><summary>Solution — Task A & B</summary>

```python
def snis_mean(p_tilde, proposal, n=20000):
    x = proposal.rvs(n, random_state=rng)
    w = p_tilde(x) / proposal.pdf(x)
    return np.sum(w * x) / np.sum(w)

def ess(w):
    return w.sum()**2 / np.sum(w**2)
# As the proposal mean/scale drifts from the target, ess(w) -> 1 and estimates get unstable.
```
</details>